# [County] County Prosecution Analysis

**Source:** [County] redacted prosecution extract
**Coverage:** [years]
**Unit note:** Each chart states whether it counts *cases* (distinct case IDs) or *charges* (charge rows). A case can have multiple charges with different outcomes.

## Setup
Imports the shared `viz_style.py` so colors, fonts, n labels, and exports match across counties.

In [ ]:
import sys, subprocess, os
repo_root = subprocess.run(["git", "rev-parse", "--show-toplevel"],
                           capture_output=True, text=True).stdout.strip()
sys.path.insert(0, repo_root)

import pandas as pd
import matplotlib.pyplot as plt
from viz_style import set_style, stacked_pct_bar, save_fig, OUTCOME_COLORS, color_for, normalize_outcome

set_style()

COUNTY = "plymouth"  # change per notebook
FIG_DIR = os.path.join(repo_root, "figures", COUNTY)
os.makedirs(FIG_DIR, exist_ok=True)

## Load and Clean Data

In [ ]:
df = pd.read_csv("PATH/TO/EXTRACT.csv")  # change path

# Rename to shared column names so the code below works for every county.
# Left side = your extract's column name.
df = df.rename(columns={
    "CASE_ID_COL":     "case_id",
    "DISPOSITION_COL": "outcome_raw",
    "DISP_DATE_COL":   "disp_date",
    "RECV_DATE_COL":   "recv_date",
    "COURT_COL":       "court",
    "COURT_LEVEL_COL": "court_level",   # District / Superior
    "CRIME_CODE_COL":  "crime_code",
    "CASE_TAG_COL":    "case_tag",      # DV, SA, etc.
})

df["outcome"] = df["outcome_raw"].map(normalize_outcome)
df["disp_date"] = pd.to_datetime(df["disp_date"], errors="coerce")
df["recv_date"] = pd.to_datetime(df["recv_date"], errors="coerce")
df["disp_year"] = df["disp_date"].dt.year
df["recv_year"] = df["recv_date"].dt.year

# Client doesn't want motor vehicle crimes. Adjust the code list if needed.
# df = df[df["crime_code"] != "MMVE"]

print(f"{len(df):,} charge rows, {df['case_id'].nunique():,} cases")
df["outcome"].value_counts()

### Case-level outcome
Charge-level counts and case-level counts answer different questions. This builds one outcome per case so we can report either.

**Rule:** Each case gets its most serious outcome across all charges: Guilty > CWOF > Other Plea > Not Guilty > Nolle Prosequi > Dismissed > Other.
- Cases with any charge still missing a disposition are **unresolved** and excluded from outcome charts (count reported below).
- Disposition year = latest charge disposition date in the case.
- Lead charge: [note whether the extract has a charge sequence / lead flag; if not, "lead charge not identifiable in the extract"]

In [ ]:
SEVERITY = ["Guilty", "CWOF", "Other Plea", "Not Guilty",
            "Nolle Prosequi", "Dismissed", "Other", "Missing"]

def case_outcome(outcomes):
    s = set(outcomes)
    if "Missing" in s and len(s) > 1:
        return "Unresolved"   # some charges still open
    for o in SEVERITY:
        if o in s:
            return o
    return "Other"

all_cases = (df.groupby("case_id")
               .agg(outcome=("outcome", case_outcome),
                    disp_year=("disp_year", "max"),
                    recv_year=("recv_year", "min"),
                    court_level=("court_level", "first"),
                    court=("court", "first"))
               .reset_index())

n_unresolved = (all_cases["outcome"] == "Unresolved").sum()
cases = all_cases[all_cases["outcome"] != "Unresolved"]
print(f"{len(cases):,} resolved cases used in outcome charts; {n_unresolved:,} unresolved excluded")
cases["outcome"].value_counts()

## Q1. Domestic Violence, Sexual Assault, and Child Abuse Dispositions Over Time

### Domestic Violence

In [ ]:
tag_cases = df.loc[df["case_tag"] == "DV", "case_id"].unique()
sub = cases[cases["case_id"].isin(tag_cases)]

counts = sub.groupby(["disp_year", "outcome"]).size().unstack(fill_value=0)
ax = stacked_pct_bar(counts, unit="cases",
                     title=f"{COUNTY.title()}: Domestic Violence case outcomes by disposition year",
                     xlabel="Disposition year")
save_fig(ax.figure, f"{FIG_DIR}/q1_dv_outcomes.png")

**How this was calculated**
- **Unit:** cases / charges
- **Denominator:** [e.g., all cases with a disposition in that year]
- **Filters:** [e.g., cases tagged DV, excludes motor vehicle offenses]
- **Date field:** received date / disposition date
- **Outcome rule:** [e.g., case counted as dismissed if all charges dismissed]

### Sexual Assault

In [ ]:
tag_cases = df.loc[df["case_tag"] == "SA", "case_id"].unique()
sub = cases[cases["case_id"].isin(tag_cases)]

counts = sub.groupby(["disp_year", "outcome"]).size().unstack(fill_value=0)
ax = stacked_pct_bar(counts, unit="cases",
                     title=f"{COUNTY.title()}: Sexual Assault case outcomes by disposition year",
                     xlabel="Disposition year")
save_fig(ax.figure, f"{FIG_DIR}/q1_sa_outcomes.png")

**How this was calculated**
- **Unit:** cases / charges
- **Denominator:** [e.g., all cases with a disposition in that year]
- **Filters:** [e.g., cases tagged DV, excludes motor vehicle offenses]
- **Date field:** received date / disposition date
- **Outcome rule:** [e.g., case counted as dismissed if all charges dismissed]

### Child Abuse

In [ ]:
tag_cases = df.loc[df["case_tag"] == "CHAB", "case_id"].unique()
sub = cases[cases["case_id"].isin(tag_cases)]

counts = sub.groupby(["disp_year", "outcome"]).size().unstack(fill_value=0)
ax = stacked_pct_bar(counts, unit="cases",
                     title=f"{COUNTY.title()}: Child Abuse case outcomes by disposition year",
                     xlabel="Disposition year")
save_fig(ax.figure, f"{FIG_DIR}/q1_chab_outcomes.png")

**How this was calculated**
- **Unit:** cases / charges
- **Denominator:** [e.g., all cases with a disposition in that year]
- **Filters:** [e.g., cases tagged DV, excludes motor vehicle offenses]
- **Date field:** received date / disposition date
- **Outcome rule:** [e.g., case counted as dismissed if all charges dismissed]

**Limitation:** [e.g., Child abuse code yields small annual counts; percentages are sensitive to a few cases.]

## Q2. Outcomes by Court

In [ ]:
counts = cases.groupby(["court", "outcome"]).size().unstack(fill_value=0)
ax = stacked_pct_bar(counts, unit="cases",
                     title=f"{COUNTY.title()}: case outcomes by court",
                     xlabel="Court")
save_fig(ax.figure, f"{FIG_DIR}/q2_court_outcomes.png")

**How this was calculated**
- **Unit:** cases / charges
- **Denominator:** [e.g., all cases with a disposition in that year]
- **Filters:** [e.g., cases tagged DV, excludes motor vehicle offenses]
- **Date field:** received date / disposition date
- **Outcome rule:** [e.g., case counted as dismissed if all charges dismissed]

**Note:** Comparisons are not adjusted for offense mix.

## Q3. Crime-Type Breakdown by Municipality and Countywide

In [ ]:
top = df["crime_code"].value_counts().head(15)
pct = top / len(df) * 100

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(pct.index[::-1], pct.values[::-1], color=OUTCOME_COLORS["Dismissed"])
for y, (v, n) in enumerate(zip(pct.values[::-1], top.values[::-1])):
    ax.text(v + 0.3, y, f"{v:.1f}% (n={n:,})", va="center", fontsize=12)
ax.set_xlabel("% of charges")
ax.grid(axis="x"); ax.grid(axis="y", visible=False)
ax.set_title(f"{COUNTY.title()}: 15 most frequent crime codes")
save_fig(fig, f"{FIG_DIR}/q3_crime_codes.png")

**How this was calculated**
- **Unit:** cases / charges
- **Denominator:** [e.g., all cases with a disposition in that year]
- **Filters:** [e.g., cases tagged DV, excludes motor vehicle offenses]
- **Date field:** received date / disposition date
- **Outcome rule:** [e.g., case counted as dismissed if all charges dismissed]

**Limitation:** [e.g., No incident municipality field, so a town-level breakdown is unavailable.]

## Q4. Victims by Race or Ethnicity Over Time

In [ ]:
# If victim race is available, use stacked_pct_bar with victim counts.
# Pass colors per race group explicitly; do not reuse OUTCOME_COLORS.

**How this was calculated**
- **Unit:** cases / charges
- **Denominator:** [e.g., all cases with a disposition in that year]
- **Filters:** [e.g., cases tagged DV, excludes motor vehicle offenses]
- **Date field:** received date / disposition date
- **Outcome rule:** [e.g., case counted as dismissed if all charges dismissed]

**Limitation:** [e.g., The extract has no victim fields, so this breakdown is unavailable.]

## Q5. Charging Patterns for the Five Requested Offense Groups

In [ ]:
groups = ["DV", "SA", "CHAB", "ELAB", "HACR"]  # adjust to your county's tags
total_by_year = df.groupby("recv_year")["case_id"].nunique()
tagged = (df[df["case_tag"].isin(groups)]
            .groupby(["recv_year", "case_tag"])["case_id"].nunique()
            .unstack(fill_value=0))

fig, ax = plt.subplots()
line_colors = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
for g, c in zip(groups, line_colors):
    if g in tagged:
        share = tagged[g] / total_by_year * 100
        ax.plot(share.index, share.values, marker="o", markersize=8, linewidth=2, color=c, label=g)
        ax.annotate(g, (share.index[-1], share.values[-1]), xytext=(8, 0),
                    textcoords="offset points", va="center", fontsize=12)
ax.set_ylabel("% of all cases received")
ax.set_xlabel("Received year")
ax.legend(loc="upper left", bbox_to_anchor=(1.08, 1))
ax.set_title(f"{COUNTY.title()}: offense groups as share of cases received")
save_fig(fig, f"{FIG_DIR}/q5_offense_groups.png")

tagged  # raw counts table for reference

**How this was calculated**
- **Unit:** cases / charges
- **Denominator:** [e.g., all cases with a disposition in that year]
- **Filters:** [e.g., cases tagged DV, excludes motor vehicle offenses]
- **Date field:** received date / disposition date
- **Outcome rule:** [e.g., case counted as dismissed if all charges dismissed]

**Limitation:** [e.g., Hate crime code needs validation; bias motivation is not recorded.]

## Q6. Nolle Prosequi by Crime Type and Court Level

**NOPR definitions**
- **At least one:** case has one or more charges marked NOPR
- **All charges:** every charge in the case is marked NOPR
- **Some but not all:** at least one NOPR, but not every charge

In [ ]:
nopr = (df.assign(is_nopr=df["outcome"] == "Nolle Prosequi")
          .groupby("case_id")
          .agg(any_nopr=("is_nopr", "any"), all_nopr=("is_nopr", "all"),
               disp_year=("disp_year", "max"), court_level=("court_level", "first"))
          .reset_index())
nopr["nopr_group"] = "No NOPR"
nopr.loc[nopr["any_nopr"] & ~nopr["all_nopr"], "nopr_group"] = "Some but not all"
nopr.loc[nopr["all_nopr"], "nopr_group"] = "All charges"

pd.crosstab(nopr["court_level"], nopr["nopr_group"], margins=True)

### NOPR cases by year and court level

In [ ]:
NOPR_COLORS = {"All charges": "#eb6834", "Some but not all": "#2a78d6"}
has = nopr[nopr["any_nopr"]]

for level in ["District", "Superior"]:
    counts = (has[has["court_level"] == level]
                .groupby(["disp_year", "nopr_group"]).size()
                .unstack(fill_value=0)
                .reindex(columns=["All charges", "Some but not all"], fill_value=0))
    fig, ax = plt.subplots()
    bottom = 0
    for col in counts.columns:
        ax.bar(counts.index, counts[col], bottom=bottom, color=NOPR_COLORS[col],
               label=col, edgecolor="#fcfcfb", linewidth=2)
        bottom = bottom + counts[col]
    for x, n in zip(counts.index, counts.sum(axis=1)):
        ax.text(x, n, f"n={n:,}", ha="center", va="bottom", fontsize=12)
    ax.set_xticks(counts.index)
    ax.set_xticklabels([str(int(y)) for y in counts.index])
    ax.set_ylabel("Cases with any NOPR")
    ax.set_xlabel("Disposition year")
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
    ax.set_title(f"{COUNTY.title()} {level} Court: NOPR cases by year")
    save_fig(fig, f"{FIG_DIR}/q6_nopr_{level.lower()}.png")

**How this was calculated**
- **Unit:** cases / charges
- **Denominator:** [e.g., all cases with a disposition in that year]
- **Filters:** [e.g., cases tagged DV, excludes motor vehicle offenses]
- **Date field:** received date / disposition date
- **Outcome rule:** [e.g., case counted as dismissed if all charges dismissed]

### NOPR by crime type

In [ ]:
nopr_charges = df[df["outcome"] == "Nolle Prosequi"]
by_crime = (nopr_charges.groupby(["court_level", "crime_code"])["case_id"].nunique()
              .rename("cases").reset_index())
by_crime["pct_of_nopr_cases"] = (by_crime["cases"]
    / by_crime.groupby("court_level")["cases"].transform("sum") * 100).round(1)
by_crime.sort_values(["court_level", "cases"], ascending=[True, False]).groupby("court_level").head(10)

**How this was calculated**
- **Unit:** cases / charges
- **Denominator:** [e.g., all cases with a disposition in that year]
- **Filters:** [e.g., cases tagged DV, excludes motor vehicle offenses]
- **Date field:** received date / disposition date
- **Outcome rule:** [e.g., case counted as dismissed if all charges dismissed]

**Note:** A case can appear under more than one crime code.

## Questions for the Client
- [e.g., Can you provide an incident municipality field?]
- [e.g., Can you confirm the codes for child abuse, elder abuse, and hate crimes?]